# Predicciones LightGBM Multi-Horizonte

Genera predicciones con LightGBM (18 horizontes) y crea la tabla `fact_predicciones` en PostgreSQL.

## Dimensiones reutilizadas del datamart
- `dim_tiempo`, `dim_riesgo`, `dim_sector`, `dim_sucursal`

## Entradas
- `output/datasets/datos_preprocesados.csv`
- Modelos: `output/modelos_lightgbm/modelo_lgbm_h*.txt`

## Salidas
- `fact_predicciones` en PostgreSQL
- `output/predicciones/predicciones.csv`

In [ ]:
import json
import logging
import os
from datetime import datetime

import lightgbm as lgb
import numpy as np
import pandas as pd
import psycopg2

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)
logger = logging.getLogger(__name__)

## Configuración

In [32]:
PATH_INPUT = "output/datasets/datos_preprocesados.csv"
PATH_LGBM = "output/modelos_lightgbm"
PATH_OUTPUT = "output/predicciones"
os.makedirs(PATH_OUTPUT, exist_ok=True)

DB_CONFIG = {
    "host": "localhost",
    "port": "5432",
    "database": "postgres_db",
    "user": "postgres_usr",
    "password": "admin123",
}

MAX_HORIZONTE = 18
VENTANA_LGBM = 6

print(f"Horizontes: {MAX_HORIZONTE}")
print(f"Ventana LGBM: {VENTANA_LGBM} meses")

Horizontes: 18
Ventana LGBM: 6 meses


## Carga de datos y modelos

In [33]:
df = pd.read_csv(PATH_INPUT)
df['mes'] = pd.to_datetime(df['mes'])

with open(os.path.join(PATH_LGBM, 'config_lgbm_18m.json')) as f:
    config_lgbm = json.load(f)

features_numericas = config_lgbm['features_numericas']

modelos_lgbm = []
for h in range(1, MAX_HORIZONTE + 1):
    modelos_lgbm.append(
        lgb.Booster(model_file=os.path.join(PATH_LGBM, f'modelo_lgbm_h{h}.txt'))
    )

print(f"Datos: {len(df):,} registros, {df['bloque_id'].nunique()} bloques")
print(f"Modelos LightGBM: {len(modelos_lgbm)}")

Datos: 20,025 registros, 950 bloques
Modelos LightGBM: 18


## Funciones

In [34]:
def crear_secuencias_lgbm(df, bloque_id, features, ventana):
    """Genera secuencias estadísticas para LightGBM."""
    df_b = df[df['bloque_id'] == bloque_id].sort_values('mes')
    if len(df_b) < ventana:
        return None, None, None

    X_seq, meses = [], []
    for i in range(len(df_b) - ventana + 1):
        hist = df_b[features].iloc[i:i+ventana]
        feats = []
        for col in features:
            v = hist[col].values
            feats.extend([
                np.mean(v), np.std(v), np.min(v), np.max(v),
                np.median(v), v[-1], v[-1] - v[0],
            ])
        X_seq.append(feats)
        meses.append(df_b['mes'].iloc[i + ventana - 1])
    return np.array(X_seq), meses, df_b


print("Funciones definidas.")

Funciones definidas.


In [35]:
# Actualizar todas las dimensiones desde el CSV
# (la MV puede no cubrir todos los registros del dataset)
conn_tmp = psycopg2.connect(
    host=DB_CONFIG['host'], port=DB_CONFIG['port'],
    dbname=DB_CONFIG['database'],
    user=DB_CONFIG['user'], password=DB_CONFIG['password'],
)
conn_tmp.autocommit = True
cur = conn_tmp.cursor()

# 1. dim_tiempo
fechas_csv = df['mes'].sort_values().unique()
for fecha in fechas_csv:
    cur.execute("""
        INSERT INTO dim_tiempo (mes, anio, trimestre, mes_del_anio, nombre_mes)
        VALUES (%s, %s, %s, %s, %s)
        ON CONFLICT (mes) DO NOTHING
    """, (fecha, fecha.year, (fecha.month - 1) // 3 + 1, fecha.month, fecha.strftime('%B')))

# 2. dim_riesgo
for riesgo in df['riesgo'].unique():
    cur.execute("""
        INSERT INTO dim_riesgo (codigo_riesgo, descripcion)
        VALUES (%s, %s) ON CONFLICT (codigo_riesgo) DO NOTHING
    """, (str(riesgo), str(riesgo)))

# 3. dim_sector
for sector in df['sector'].unique():
    cur.execute("""
        INSERT INTO dim_sector (codigo_sector, descripcion)
        VALUES (%s, %s) ON CONFLICT (codigo_sector) DO NOTHING
    """, (str(sector), str(sector)))

# 4. dim_sucursal
for suc in df['codigo_sucursal'].unique():
    cur.execute("""
        INSERT INTO dim_sucursal (codigo_sucursal, codigo_provincia)
        VALUES (%s, 0) ON CONFLICT (codigo_sucursal) DO NOTHING
    """, (int(suc),))

cur.close()
conn_tmp.close()
print("Dimensiones actualizadas desde CSV.")
print(f"  Fechas: {len(fechas_csv)}, Riesgos: {df['riesgo'].nunique()}, "
      f"Sectores: {df['sector'].nunique()}, Sucursales: {df['codigo_sucursal'].nunique()}")

Dimensiones actualizadas desde CSV.
  Fechas: 48, Riesgos: 1, Sectores: 20, Sucursales: 64


## Generar predicciones

In [ ]:
predicciones = []

for bloque in df['bloque_id'].unique():
    info = df[df['bloque_id'] == bloque]
    riesgo = info['riesgo'].iloc[0]
    sector = info['sector'].iloc[0]
    sucursal = int(info['codigo_sucursal'].iloc[0])

    X_seq, meses_seq, _ = crear_secuencias_lgbm(
        df, bloque, features_numericas, VENTANA_LGBM
    )
    if X_seq is None or len(X_seq) == 0:
        continue

    for idx in range(len(X_seq)):
        row = {
            'bloque_id': bloque, 'riesgo': riesgo, 'sector': sector,
            'codigo_sucursal': sucursal, 'mes_prediccion': meses_seq[idx],
        }
        x_in = X_seq[idx:idx+1]
        for h in range(MAX_HORIZONTE):
            p = float(modelos_lgbm[h].predict(x_in)[0])
            row[f'prob_h{h+1}'] = round(p, 6)
            row[f'pred_h{h+1}'] = int(p > 0.5)
        predicciones.append(row)

df_pred = pd.DataFrame(predicciones)

prob_cols = [f'prob_h{i+1}' for i in range(MAX_HORIZONTE)]
pred_cols = [f'pred_h{i+1}' for i in range(MAX_HORIZONTE)]

df_pred['prob_media'] = df_pred[prob_cols].mean(axis=1).round(6)
df_pred['pred_media'] = (df_pred['prob_media'] > 0.5).astype(int)
df_pred['crisis_count'] = df_pred[pred_cols].sum(axis=1).astype(int)
df_pred['fecha_ejecucion'] = datetime.now()

print(f"Predicciones: {len(df_pred):,}")
print(f"\nDistribucion pred_media:")
print(df_pred['pred_media'].value_counts().to_string())

In [ ]:
csv_path = os.path.join(PATH_OUTPUT, 'predicciones.csv')
df_pred.to_csv(csv_path, index=False)
print(f"CSV guardado: {csv_path}")

## Crear tabla `fact_predicciones`

In [ ]:
conn = psycopg2.connect(
    host=DB_CONFIG['host'], port=DB_CONFIG['port'],
    dbname=DB_CONFIG['database'],
    user=DB_CONFIG['user'], password=DB_CONFIG['password'],
)
conn.autocommit = True

DDL = """
DROP TABLE IF EXISTS fact_predicciones CASCADE;

CREATE TABLE fact_predicciones (
    id_prediccion    SERIAL PRIMARY KEY,
    id_tiempo        INTEGER NOT NULL REFERENCES dim_tiempo(id_tiempo),
    id_riesgo        INTEGER NOT NULL REFERENCES dim_riesgo(id_riesgo),
    id_sector        INTEGER NOT NULL REFERENCES dim_sector(id_sector),
    id_sucursal      INTEGER NOT NULL REFERENCES dim_sucursal(id_sucursal),
    bloque_id        VARCHAR(50) NOT NULL,

    prob_h01  NUMERIC(10,6), prob_h02  NUMERIC(10,6), prob_h03  NUMERIC(10,6),
    prob_h04  NUMERIC(10,6), prob_h05  NUMERIC(10,6), prob_h06  NUMERIC(10,6),
    prob_h07  NUMERIC(10,6), prob_h08  NUMERIC(10,6), prob_h09  NUMERIC(10,6),
    prob_h10  NUMERIC(10,6), prob_h11  NUMERIC(10,6), prob_h12  NUMERIC(10,6),
    prob_h13  NUMERIC(10,6), prob_h14  NUMERIC(10,6), prob_h15  NUMERIC(10,6),
    prob_h16  NUMERIC(10,6), prob_h17  NUMERIC(10,6), prob_h18  NUMERIC(10,6),

    pred_h01  INTEGER, pred_h02  INTEGER, pred_h03  INTEGER,
    pred_h04  INTEGER, pred_h05  INTEGER, pred_h06  INTEGER,
    pred_h07  INTEGER, pred_h08  INTEGER, pred_h09  INTEGER,
    pred_h10  INTEGER, pred_h11  INTEGER, pred_h12  INTEGER,
    pred_h13  INTEGER, pred_h14  INTEGER, pred_h15  INTEGER,
    pred_h16  INTEGER, pred_h17  INTEGER, pred_h18  INTEGER,

    prob_media     NUMERIC(10,6),
    pred_media     INTEGER,
    crisis_count   INTEGER,
    fecha_ejecucion TIMESTAMP,

    UNIQUE (id_tiempo, id_riesgo, id_sector, id_sucursal)
);

CREATE INDEX idx_fp_tiempo   ON fact_predicciones (id_tiempo);
CREATE INDEX idx_fp_riesgo   ON fact_predicciones (id_riesgo);
CREATE INDEX idx_fp_sector   ON fact_predicciones (id_sector);
CREATE INDEX idx_fp_sucursal ON fact_predicciones (id_sucursal);
CREATE INDEX idx_fp_bloque   ON fact_predicciones (bloque_id);
CREATE INDEX idx_fp_pred     ON fact_predicciones (pred_media);
"""

cur = conn.cursor()
cur.execute(DDL)
print("Tabla fact_predicciones creada.")
cur.close()

## Poblar tabla

In [ ]:
SQL_INSERT = """
INSERT INTO fact_predicciones (
    id_tiempo, id_riesgo, id_sector, id_sucursal,
    bloque_id,
    prob_h01, prob_h02, prob_h03, prob_h04, prob_h05, prob_h06,
    prob_h07, prob_h08, prob_h09, prob_h10, prob_h11, prob_h12,
    prob_h13, prob_h14, prob_h15, prob_h16, prob_h17, prob_h18,
    pred_h01, pred_h02, pred_h03, pred_h04, pred_h05, pred_h06,
    pred_h07, pred_h08, pred_h09, pred_h10, pred_h11, pred_h12,
    pred_h13, pred_h14, pred_h15, pred_h16, pred_h17, pred_h18,
    prob_media, pred_media, crisis_count, fecha_ejecucion
) VALUES (
    (SELECT id_tiempo FROM dim_tiempo WHERE mes = %s),
    (SELECT id_riesgo FROM dim_riesgo WHERE codigo_riesgo = %s),
    (SELECT id_sector FROM dim_sector WHERE codigo_sector = %s),
    (SELECT id_sucursal FROM dim_sucursal WHERE codigo_sucursal = %s),
    %s,
    %s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,
    %s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,
    %s, %s, %s, %s
)
ON CONFLICT (id_tiempo, id_riesgo, id_sector, id_sucursal)
DO UPDATE SET
    bloque_id = EXCLUDED.bloque_id,
    prob_h01 = EXCLUDED.prob_h01, prob_h02 = EXCLUDED.prob_h02,
    prob_h03 = EXCLUDED.prob_h03, prob_h04 = EXCLUDED.prob_h04,
    prob_h05 = EXCLUDED.prob_h05, prob_h06 = EXCLUDED.prob_h06,
    prob_h07 = EXCLUDED.prob_h07, prob_h08 = EXCLUDED.prob_h08,
    prob_h09 = EXCLUDED.prob_h09, prob_h10 = EXCLUDED.prob_h10,
    prob_h11 = EXCLUDED.prob_h11, prob_h12 = EXCLUDED.prob_h12,
    prob_h13 = EXCLUDED.prob_h13, prob_h14 = EXCLUDED.prob_h14,
    prob_h15 = EXCLUDED.prob_h15, prob_h16 = EXCLUDED.prob_h16,
    prob_h17 = EXCLUDED.prob_h17, prob_h18 = EXCLUDED.prob_h18,
    pred_h01 = EXCLUDED.pred_h01, pred_h02 = EXCLUDED.pred_h02,
    pred_h03 = EXCLUDED.pred_h03, pred_h04 = EXCLUDED.pred_h04,
    pred_h05 = EXCLUDED.pred_h05, pred_h06 = EXCLUDED.pred_h06,
    pred_h07 = EXCLUDED.pred_h07, pred_h08 = EXCLUDED.pred_h08,
    pred_h09 = EXCLUDED.pred_h09, pred_h10 = EXCLUDED.pred_h10,
    pred_h11 = EXCLUDED.pred_h11, pred_h12 = EXCLUDED.pred_h12,
    pred_h13 = EXCLUDED.pred_h13, pred_h14 = EXCLUDED.pred_h14,
    pred_h15 = EXCLUDED.pred_h15, pred_h16 = EXCLUDED.pred_h16,
    pred_h17 = EXCLUDED.pred_h17, pred_h18 = EXCLUDED.pred_h18,
    prob_media = EXCLUDED.prob_media,
    pred_media = EXCLUDED.pred_media,
    crisis_count = EXCLUDED.crisis_count,
    fecha_ejecucion = EXCLUDED.fecha_ejecucion;
"""

cur = conn.cursor()
inserted = 0

for _, row in df_pred.iterrows():
    riesgo_s = str(row['riesgo'])
    sector_s = str(row['sector'])
    sucursal_i = int(row['codigo_sucursal'])

    params = (
        row['mes_prediccion'], riesgo_s, sector_s, sucursal_i,
        row['bloque_id'],
    )
    for h in range(1, MAX_HORIZONTE + 1):
        params += (row[f'prob_h{h}'],)
    for h in range(1, MAX_HORIZONTE + 1):
        params += (int(row[f'pred_h{h}']),)
    params += (row['prob_media'], int(row['pred_media']),
              int(row['crisis_count']), row['fecha_ejecucion'])

    cur.execute(SQL_INSERT, params)
    inserted += cur.rowcount

conn.commit()
cur.close()
print(f"Filas insertadas/actualizadas: {inserted:,}")

## Validación

In [ ]:
print("=" * 60)
print("VALIDACION DE FACT_PREDICCIONES")
print("=" * 60)

cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM fact_predicciones")
print(f"\nTotal registros: {cur.fetchone()[0]:,}")

cur.execute("SELECT pred_media, COUNT(*) FROM fact_predicciones GROUP BY pred_media ORDER BY pred_media")
print("\nDistribucion pred_media:")
for r in cur.fetchall():
    label = "Sin crisis" if r[0] == 0 else "Crisis"
    print(f"  {label}: {r[1]:,}")

cur.execute("""
    SELECT dr.descripcion, COUNT(*) 
    FROM fact_predicciones fp JOIN dim_riesgo dr ON fp.id_riesgo = dr.id_riesgo 
    GROUP BY dr.descripcion ORDER BY dr.descripcion
""")
print("\nPor riesgo:")
for r in cur.fetchall():
    print(f"  {r[0]}: {r[1]:,}")

cur.execute("""
    SELECT dt.mes, 
    ROUND(AVG(fp.prob_media)::numeric, 4), 
    SUM(fp.pred_media) 
    FROM fact_predicciones fp 
    JOIN dim_tiempo dt ON fp.id_tiempo = dt.id_tiempo 
    GROUP BY dt.mes ORDER BY dt.mes LIMIT 10
""")
print("\nMuestra por mes:")
print(f"{'mes':<12} {'prob_media':>12} {'crisis_pred':>12}")
print("-" * 40)
for r in cur.fetchall():
    print(f"{str(r[0]):<12} {r[1]:>12} {r[2]:>12}")

cur.close()
print("\n" + "=" * 60)

In [ ]:
cur = conn.cursor()
cur.execute("""
    SELECT bloque_id,
           ROUND(prob_media::numeric, 4) AS prob,
           pred_media,
           crisis_count
    FROM fact_predicciones
    ORDER BY prob_media DESC
    LIMIT 10
""")
print("Top 10 mayor probabilidad de crisis:")
print(f"{'bloque_id':<15} {'prob':>8} {'pred':>6} {'#crisis':>8}")
print("-" * 40)
for r in cur.fetchall():
    print(f"{r[0]:<15} {r[1]:>8} {r[2]:>6} {r[3]:>8}")
cur.close()

In [ ]:
conn.close()
print("Conexion cerrada.")

![icon](../../DocumentosBase/yachayCuadrado.jpg)<br/>***<omar.velez@yachaytech.edu.ec>***<br/>*julio 2026*